**Load packages and dataset**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = pd.read_csv("../data/GiveMeSomeCredit/cs-training.csv")

**Data pre-processing**

*Pre-filtering of features*

In [2]:
features = [col for col in df if (col not in ["SeriousDlqin2yrs", "Index", "NumberOfDependents", "age"])] # extract features except for target variable, index, and number of dependents and age (due to fairness)

df.head(10) # display dataframe for overview

,Index,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0
5,6,0,0.213179,74,0,0.375607,3500.0,3,0,1,0,1.0
6,7,0,0.305682,57,0,5710.000000,NaN,8,0,3,0,0.0
7,8,0,0.754464,39,0,0.209940,3500.0,8,0,0,0,0.0
8,9,0,0.116951,27,0,46.000000,NaN,2,0,0,0,NaN
9,10,0,0.189169,57,0,0.606291,23684.0,9,0,4,0,2.0


*Treating missing values*

In [3]:
# Drop rows where values are NaN before splitting
print(df.isnull().sum()) # note: of the pre-filtered features, only MonthlyIncome has missing values
df_cleaned = df.dropna(subset=['SeriousDlqin2yrs']).fillna(-1) #set missing values in MonthlyIncome to -1

Index                                       0
SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64


*Train/Test Split*

In [4]:
X = df_cleaned[features]
y = df_cleaned["SeriousDlqin2yrs"]

X_train, X_test, y_train, y_test = train_test_split(   # current split 70:30 -> might try variations?
    X, y, test_size=0.3, random_state=42, stratify=y
)

min_instances_per_bin = np.ceil(X_train.shape[0]*0.05)

print("Required minimum number of instances per Bin (5 percent rule): ", min_instances_per_bin)

Required minimum number of instances per Bin (5 percent rule):  5250.0


**Initial equal-frequency binning**

*Helper functions*

In [5]:
def _calculate_woe_components(grouped_df):
    """
    Calculates Weight of Evidence (WoE) and Information Value (IV) components.
    """
    # Calculate percentages with small epsilon to avoid division by zero
    eps = 1e-6
    grouped_df['Pct Goods dec'] = grouped_df['Goods'] / (grouped_df['Goods'].sum() + eps)
    grouped_df['Pct Bads dec'] = grouped_df['Bads'] / (grouped_df['Bads'].sum() + eps)
    
    # Calculate Bad Rate and WoE
    grouped_df['Bad Rate'] = grouped_df['Bads'] / (grouped_df['Count'] + eps)
    grouped_df['WoE'] = np.log((grouped_df['Pct Goods dec'] + eps) / (grouped_df['Pct Bads dec'] + eps))
    
    return grouped_df

def _format_percentages(grouped_df, columns):
    """
    Converts decimal percentages to formatted percentage strings.
    """
    for col in columns:
        grouped_df[col] = (grouped_df[col] * 100).round(2).astype(str) + '%'
    return grouped_df

*Functions for table visualization of binnings*

In [6]:
def _binning_table_core(X, y, binned_col, feature):
    """
    Calculates IV and table overview including bin WoEs and Bad Rates for an already binned feature.
    """
    df = X.copy()
    df['target'] = y.values

    # Aggregate data by bins
    grouped = df.groupby(binned_col).agg(
        Count=(feature, 'count'),
        Goods=('target', lambda x: (x == 0).sum()),
        Bads=('target', lambda x: (x == 1).sum())
    ).reset_index()

    # Calculate WoE and IV metrics
    grouped = _calculate_woe_components(grouped)
    IV = ((grouped['Pct Goods dec'] - grouped['Pct Bads dec']) * grouped['WoE']).sum()
    print(f"\nInformation Value (IV) for '{feature}': {IV:.4f}")

    # Format output percentages
    grouped = _format_percentages(grouped, ['Pct Goods dec', 'Pct Bads dec', 'Bad Rate'])
    grouped.rename(columns={
        'Pct Goods dec': 'Pct Goods',
        'Pct Bads dec': 'Pct Bads'
    }, inplace=True)
    
    # Round WoE values
    grouped['WoE'] = grouped['WoE'].round(2)

    # Return final column order
    return grouped[[binned_col, 'Count', 'Goods', 'Pct Goods', 'Bads', 'Pct Bads', 'Bad Rate', 'WoE']]

*Initial equal-frequency binning*

In [7]:
def binning_table_n_bins(X, y, feature, n_bins):
    """
    Bins a feature equally into n_bins bins, with missing values (-1) in a separate bin on top.
    """
    X = X.copy()
    binned_col = f'{feature}_bin'
    X[binned_col] = None

    # treat MonthlyIncome missing values (encoded as -1) as separate bin
    X.loc[X[feature] == -1, binned_col] = 'Missing'

    # perform equal-frequency binning only for all other values
    mask = X[feature] != -1
    X.loc[mask, binned_col] = pd.qcut(X.loc[mask, feature], q=n_bins, duplicates='drop')

    # make table
    grouped = _binning_table_core(X, y, binned_col, feature)
    grouped = grouped.sort_values(by=binned_col, key=lambda x: x == 'Missing', ascending=False)
    grouped.reset_index(drop=True, inplace=True)
    
    return grouped

In [8]:
def plot_distribution(df, feature):
    df = df.copy()
    data = df[feature]
    min_val = data.min()
    max_val = data.max()

    plt.hist(df[feature], bins=20, edgecolor='k')
    plt.xlabel(f'{feature}')
    plt.ylabel('Count')
    plt.title(f'{feature} Distribution')
    plt.xlim(min_val, max_val)
    plt.show()

for col in features:
    print(binning_table_n_bins(X_train, y_train, col, n_bins=6))


Information Value (IV) for 'RevolvingUtilizationOfUnsecuredLines': 1.0767
  RevolvingUtilizationOfUnsecuredLines_bin  Count  Goods Pct Goods  Bads  \
0                         (-0.001, 0.0129]  17500  17146     17.5%   354   
1                         (0.0129, 0.0543]  17500  17238    17.59%   262   
2                          (0.0543, 0.154]  17500  17105    17.46%   395   
3                           (0.154, 0.381]  17500  16814    17.16%   686   
4                           (0.381, 0.801]  17500  15866    16.19%  1634   
5                         (0.801, 50708.0]  17500  13813     14.1%  3687   

  Pct Bads Bad Rate   WoE  
0    5.04%    2.02%  1.24  
1    3.73%     1.5%  1.55  
2    5.63%    2.26%  1.13  
3    9.77%    3.92%  0.56  
4   23.28%    9.34% -0.36  
5   52.54%   21.07% -1.32  

Information Value (IV) for 'NumberOfTime30-59DaysPastDueNotWorse': 0.0000
  NumberOfTime30-59DaysPastDueNotWorse_bin   Count  Goods Pct Goods  Bads  \
0                           (-0.001, 98.0]  

**Iterative manual binning**

*Function for table visualization*

In [9]:
def binning_table_splitpoints(X, y, feature, splitpoints):
    """
    Bins a feature along given splitpoints.
    """
    X = X.copy()
    binned_col = f'{feature}_bin'
    bins = [-np.inf] + list(splitpoints) + [np.inf]
    labels = [f'({bins[i]}, {bins[i+1]}]' for i in range(len(bins)-1)]
    X[binned_col] = pd.cut(X[feature], bins=bins, labels=labels, include_lowest=True)
    return _binning_table_core(X, y, binned_col, feature)

*Manual definition of split points*

In [10]:
# manually define splitpoints for a new binning based on the information attained from the indicator tables
# repeat iteratively, based on resulting tables, maximizing IV under consideration of constraints

splitpoints_dict = {
    'RevolvingUtilizationOfUnsecuredLines' : [0.1, 0.3, 0.6, 0.9], # all constraints fulfilled
    'NumberOfTime30-59DaysPastDueNotWorse' : [0, 1],  # all constraints fulfilled
    'DebtRatio' : [0.3, 0.5], # all constraints fulfilled
    'MonthlyIncome' : [-1, 3000, 5000, 7000, 10000], # except missing value -1 from monotonicity contraint (reasonable for this feature, based on domain knowledge)
     #'NumberOfOpenCreditLinesAndLoans' : [],  # exclude due to unintuitive/not easily explainable behavior
    'NumberOfTimes90DaysLate' : [0], # only one split possible due to 5% contraint, all constraints fulfilled
    'NumberRealEstateLoansOrLines' : [0, 2, 3], # except 0 from monotonicity contraint (reasonable for this feature, based on domain knowledge)
    'NumberOfTime60-89DaysPastDueNotWorse' : [0] # only one split possible due to 5% contraint, all constraints fulfilled
}

In [11]:
# split along the points as given in the dictionary

for col in features:
    if col not in splitpoints_dict:
        print(f"Warning: {col} not in splitpoints_dicts. Is this intended?")
        continue
    print(binning_table_splitpoints(X_train, y_train, col, splitpoints_dict[col]))


Information Value (IV) for 'RevolvingUtilizationOfUnsecuredLines': 1.0905
  RevolvingUtilizationOfUnsecuredLines_bin  Count  Goods Pct Goods  Bads  \
0                              (-inf, 0.1]  45182  44347    45.26%   835   
1                               (0.1, 0.3]  19780  19193    19.59%   587   
2                               (0.3, 0.6]  15343  14308     14.6%  1035   
3                               (0.6, 0.9]  10707   9267     9.46%  1440   
4                               (0.9, inf]  13988  10867    11.09%  3121   

  Pct Bads Bad Rate   WoE  
0    11.9%    1.85%  1.34  
1    8.36%    2.97%  0.85  
2   14.75%    6.75% -0.01  
3   20.52%   13.45% -0.77  
4   44.47%   22.31% -1.39  

Information Value (IV) for 'NumberOfTime30-59DaysPastDueNotWorse': 0.7245
  NumberOfTime30-59DaysPastDueNotWorse_bin  Count  Goods Pct Goods  Bads  \
0                                (-inf, 0]  88178  84618    86.36%  3560   
1                                   (0, 1]  11252   9592     9.79%  1660 

**Manual feature selection**

In [12]:
# choose features to use
# our approach: select all features with IV > 0.02 (based on rule of thumb from Siddiqi, 2006)
# but exclude NumberOfOpenCreditLinesOrLoans due to unintuitive / not easily explainable behavior
selected_features = ['RevolvingUtilizationOfUnsecuredLines', 
                     'NumberOfTimes90DaysLate', 
                     'NumberOfTime30-59DaysPastDueNotWorse', 
                     'NumberOfTime60-89DaysPastDueNotWorse', 
                     'MonthlyIncome', 
                     'NumberRealEstateLoansOrLines']

**WoE transformation**

In [13]:
def woe_table(series, y, thresholds):
    """
    Calculate WoE values for given thresholds, returns table of values.
    """
    bins = [-np.inf] + thresholds + [np.inf]
    binned = pd.cut(series, bins=bins)

    df_bin = pd.DataFrame({"bin": binned, "target": y})
    grouped = df_bin.groupby("bin", observed=False)["target"] 

    good = grouped.count() - grouped.sum()
    bad = grouped.sum()
    
    # Use epsilon to prevent division by 0 or log(0)
    eps = 1e-6
    good_calc = good + eps
    bad_calc = bad + eps

    # Calculate WoE
    woe = np.log(
        (good_calc / good_calc.sum()) /
        (bad_calc / bad_calc.sum())
    )

    # report original good and bad values
    table = pd.DataFrame({
        "bin": woe.index.astype(str),
        "good": good.values,
        "bad": bad.values,
        "woe": woe.values
    })

    # Optional: Warn if significant zero counts were present
    if (good < eps).any() or (bad < eps).any():
        print("Note: Near-zero counts detected, WoE values smoothed with epsilon correction")
    
    return table

In [14]:
woe_tables = {}
for col in selected_features:
    if col not in splitpoints_dict:
        print(f"Warning: {col} not in splitpoints_dicts. Is this intended?")
        continue
    woe_tables[col] = woe_table(
        X_train[col], y_train, splitpoints_dict[col]
    )

In [15]:
def apply_woe(series, woe_table, feature_thresholds):
    """
    Apply WoE transformation to a series using provided WoE table.
    """
    mapping = dict(zip(woe_table["bin"], woe_table["woe"]))
    bins = [-np.inf] + feature_thresholds + [np.inf]
    binned_series = pd.cut(series, bins=bins).astype(str)
    return binned_series.map(mapping)

X_train_woe = pd.DataFrame()
X_test_woe = pd.DataFrame()

for col in selected_features:
    if col not in splitpoints_dict:
        print(f"Warning: {col} not in splitpoints_dicts. Is this intended?")
        continue
    X_train_woe[col] = apply_woe(X_train[col], woe_tables[col], splitpoints_dict[col])
    X_test_woe[col] = apply_woe(X_test[col], woe_tables[col], splitpoints_dict[col])


**Final Regression Model**

In [16]:
final_model = LogisticRegression(solver="liblinear")
final_model.fit(X_train_woe[selected_features], y_train)

test_auc = roc_auc_score(
    y_test,
    final_model.predict_proba(X_test_woe[selected_features])[:, 1]
)
test_prec = average_precision_score(
    y_test,
    final_model.predict_proba(X_test_woe[selected_features])[:, 1]
)

print("Results final model:")
print("Test AUC:", round(test_auc, 4))
print("Test av_prec:", round(test_prec, 4))

Results final model:
Test AUC: 0.8502
Test av_prec: 0.3652


**Score Card Construction**

In [17]:
def build_scorecard(selected_features, model, woe_tables, pdo=20, base_score=600):
    """
    Build credit scorecard from logistic regression model and WoE tables.
    """
    # Calculate scaling factor
    B = pdo / np.log(2)
    
    scorecard = {}
    for i, feature in enumerate(selected_features):
        coefficient = model.coef_[0][i]
        table = woe_tables[feature].copy().round(3)
        
        # Calculate points (negative because higher WoE = lower risk = higher score)
        table["points"] = (-coefficient * table["woe"] * B).round(0).astype(int)
        scorecard[feature] = table[["bin", "woe", "points"]]
    
    return scorecard

def display_scorecard(scorecard):
    """
    Display scorecard in a formatted way.
    
    Parameters:
    scorecard: Dictionary of scorecards returned by build_scorecard
    """
    for feature, table in scorecard.items():
        print(f"\nScorecard for {feature}")
        print(table)

In [18]:
scorecard = build_scorecard(selected_features, final_model, woe_tables, pdo=20)
display_scorecard(scorecard)


Scorecard for RevolvingUtilizationOfUnsecuredLines
           bin    woe  points
0  (-inf, 0.1]  1.336      26
1   (0.1, 0.3]  0.851      16
2   (0.3, 0.6] -0.010       0
3   (0.6, 0.9] -0.774     -15
4   (0.9, inf] -1.389     -27

Scorecard for NumberOfTimes90DaysLate
           bin    woe  points
0  (-inf, 0.0]  0.396       6
1   (0.0, inf] -2.302     -35

Scorecard for NumberOfTime30-59DaysPastDueNotWorse
           bin    woe  points
0  (-inf, 0.0]  0.532       8
1   (0.0, 1.0] -0.882     -14
2   (1.0, inf] -1.895     -29

Scorecard for NumberOfTime60-89DaysPastDueNotWorse
           bin    woe  points
0  (-inf, 0.0]  0.290       3
1   (0.0, inf] -2.081     -25

Scorecard for MonthlyIncome
                 bin    woe  points
0       (-inf, -1.0]  0.211       2
1     (-1.0, 3000.0] -0.334      -3
2   (3000.0, 5000.0] -0.227      -2
3   (5000.0, 7000.0]  0.009       0
4  (7000.0, 10000.0]  0.231       2
5     (10000.0, inf]  0.464       5

Scorecard for NumberRealEstateLoansOrLines
